# Reusable cluster models — fit on Young, predict on Old

This notebook shows the full workflow of ADR-015:

1. build **one frozen reference space** (z-score + PCA) on the whole dataset (Young + Old);
2. take **subsets** (by condition, then the 25% smallest mitochondria of each subject);
3. **fit** a clustering (KMeans or GMM, k chosen automatically up to `max_clusters`) on the Young subset;
4. **save** the model under a name, **load** it back;
5. **predict** the Old subset with the same model (no refit) and **compare** cluster proportions per subject;
6. do the same for AL vs IF, and for a dataset loaded separately.

Every function prints a QC block: read it — it tells you whether the same scaler was used, how far the new dataset drifts from the training set, and how many mitochondria the model describes poorly.

In [ ]:
# ── 0. Installation & data access ────────────────────────────────────────────
# In Google Colab:
# !pip install -q git+https://github.com/<org>/mito-marker.git   # adapt the repository URL
# from google.colab import drive
# drive.mount("/content/drive")

import matplotlib.pyplot as plt
from mito_marker import (
    ingest_tem_folder, get_default_preprocessing_config, run_preprocessing,
    compute_pca, compute_umap,
    subset_by_obs_values, subset_by_feature_values, get_subset_history,
    fit_cluster_model, predict_cluster_model, project_to_reference_space,
    save_cluster_model, load_cluster_model, describe_cluster_model,
    plot_cluster_model_selection, plot_cluster_radar, plot_cluster_embedding,
    compute_cluster_proportions, plot_cluster_proportions, compare_cluster_proportions,
)

DATA_DIRECTORY_PATH = "/content/drive/MyDrive/mito-marker/data/raw/tem/"   # Codespaces: "data/raw/tem/"
MODEL_DIRECTORY_PATH = "/content/drive/MyDrive/mito-marker/models/"         # Codespaces: "data/processed/models/"
SUBJECT_COLUMN = "unique_subject_ID"   # unique across species (subject_ID repeats between species)

In [ ]:
tem_morphology_anndata = ingest_tem_folder(DATA_DIRECTORY_PATH)
# Alternative: tem_morphology_anndata = anndata.read_h5ad("data/processed/TEM All Species/_TEM Mouse.h5ad")

## 1. Reference space — ONCE, on the whole dataset

The z-score and the PCA are fitted on **every** mitochondrion (Young + Old). Their parameters are frozen in `.uns` with a fingerprint. Every subset taken afterwards inherits them.

⚠ Do **not** re-run the normalization or the PCA on a subset: that would create a space specific to the subset (the fit QC warns if it happens).

In [ ]:
# Features that must never enter a PCA or a clustering on this dataset:
#   - EXP-012: Intensity_SD_percent(_CORR) are ill-conditioned (SD / mean with a mean near 0);
#   - EXP-019: CristaeOrientation_Major/Minor/Area are quantised (they encode the analysis window).
# Flagging them non-analytical keeps them in .X but out of the PCA and the clustering.
EXCLUDED_FEATURES = [
    "Intensity_SD_percent", "Intensity_SD_percent_CORR",
    "CristaeOrientation_Major", "CristaeOrientation_Minor", "CristaeOrientation_Area",
]
tem_morphology_anndata.var.loc[EXCLUDED_FEATURES, "is_non_analytical"] = True

In [ ]:
preprocessing_config = get_default_preprocessing_config()   # transform="none", normalization="zscore_col"
tem_morphology_anndata = run_preprocessing(tem_morphology_anndata, preprocessing_config)
tem_morphology_anndata = compute_pca(tem_morphology_anndata, n_components=10)
# Optional — only to cluster / display in UMAP space:
# tem_morphology_anndata = compute_umap(tem_morphology_anndata, n_pca_components=10,
#                                       embed_all_events=True, keep_reducer=True)

## 2. Subsets

`subset_by_feature_values()` modes: `lowest_fraction`, `highest_fraction` (with `fraction=`), `between`, `below`, `above` (with `lower_value=` / `upper_value=`, in raw units).
`within_group=SUBJECT_COLUMN` takes the fraction **inside each subject**, so every subject contributes the same share of its own mitochondria.

In [ ]:
young_anndata = subset_by_obs_values(tem_morphology_anndata, {"condition": ["Young"]})
old_anndata   = subset_by_obs_values(tem_morphology_anndata, {"condition": ["Old"]})

young_small_anndata = subset_by_feature_values(
    young_anndata, feature_name="Mito_Area", mode="lowest_fraction",
    fraction=0.25, within_group=SUBJECT_COLUMN,
)
# Other modes:
#   mode="highest_fraction", fraction=0.08
#   mode="between", lower_value=0.05, upper_value=0.20
#   mode="below", upper_value=0.10      /     mode="above", lower_value=0.30

## 3. Fit on the Young subset

In [ ]:
young_small_anndata, small_mito_model = fit_cluster_model(
    training_anndata      = young_small_anndata,
    model_name            = "young_small_mito_pca_kmeans",
    algorithm             = "kmeans",           # or "gmm"
    clustering_space      = "pca",              # "scaled" | "pca" | "umap"
    n_pca_components_used = 10,
    max_clusters          = 8,                  # k tested from 2 to 8
    weight_by             = [SUBJECT_COLUMN],   # every subject weighs the same (ADR-011)
)

In [ ]:
plot_cluster_model_selection(small_mito_model)
plot_cluster_radar(young_small_anndata, small_mito_model, nest_aggregate_by=SUBJECT_COLUMN)
plot_cluster_embedding(young_small_anndata, small_mito_model)

In [ ]:
young_proportions = compute_cluster_proportions({"Young small": young_small_anndata}, small_mito_model)
plot_cluster_proportions(young_proportions, small_mito_model, relative_to="subset")            # % of the small Young mitochondria
plot_cluster_proportions(young_proportions, small_mito_model, relative_to="total_population")  # % of ALL Young mitochondria

## 4. Save the model under its name

In [ ]:
model_file_path = MODEL_DIRECTORY_PATH + "young_small_mito_pca_kmeans.joblib"
save_cluster_model(small_mito_model, model_file_path, overwrite=True)

## 5. Later / in another notebook: load and predict on Old

Two ways to define "small" Old mitochondria:
- **Option A** — the 25% smallest of each Old subject (their own quantile);
- **Option B** — every Old mitochondrion below the **same absolute size** as the Young cut.

In [ ]:
small_mito_model = load_cluster_model(model_file_path)
describe_cluster_model(small_mito_model)

# Option A
old_small_anndata = subset_by_feature_values(
    old_anndata, "Mito_Area", mode="lowest_fraction", fraction=0.25, within_group=SUBJECT_COLUMN)
# Option B
# young_threshold = get_subset_history(young_small_anndata)[-1]["upper_value_applied"]
# old_small_anndata = subset_by_feature_values(old_anndata, "Mito_Area", mode="below", upper_value=young_threshold)

old_small_anndata = predict_cluster_model(old_small_anndata, small_mito_model)
plot_cluster_embedding(old_small_anndata, small_mito_model)

## 6. Young vs Old

In [ ]:
age_datasets = {"Young": young_small_anndata, "Old": old_small_anndata}
age_proportions = compute_cluster_proportions(age_datasets, small_mito_model)
plot_cluster_proportions(age_proportions, small_mito_model, relative_to="subset")
display(compare_cluster_proportions(age_datasets, small_mito_model))   # per-subject statistics, n = subjects

## 7. Same logic for AL vs IF (or any .obs condition)

In [ ]:
if "diet" in tem_morphology_anndata.obs.columns:
    diet_datasets = {
        diet: predict_cluster_model(subset_by_obs_values(tem_morphology_anndata, {"diet": [diet]}), small_mito_model)
        for diet in ["AL", "IF"]
    }
    plot_cluster_proportions(compute_cluster_proportions(diet_datasets, small_mito_model), small_mito_model)
    display(compare_cluster_proportions(diet_datasets, small_mito_model))

## 8. A dataset loaded separately (another experiment)

`predict_cluster_model()` sees that its fingerprints differ and projects it from raw `.X` with the frozen reference parameters — no refit.

In [ ]:
# new_experiment_anndata = ingest_tem_folder("/content/drive/MyDrive/.../new_experiment/")
# new_experiment_anndata = predict_cluster_model(new_experiment_anndata, small_mito_model)